In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import product

# ============================================================
# Settings
# ============================================================

base_dir = Path(".")   # change to Path("counts") if your files are inside counts/

padj_cut = 0.05
lfc_cut = 1            # DESeq2 log2FoldChange threshold

# Your DESeq2 annotated files
# This script accepts names with or without .csv
deg_files = {
    "NSC": "NSC_vs_Mock_DE_annotated",
    "NSC+SA": "NSCSA_vs_Mock_DE_annotated",
    "SA": "SA_vs_Mock_DE_annotated",
    "V1/C3dsR": "Spec_vs_Mock_DE_annotated",
    "V1/C3dsR+SA": "SpecSA_vs_Mock_DE_annotated"
}

# Order shown in the UpSet plot
set_order = [
    "SA",
    "NSC",
    "NSC+SA",
    "V1/C3dsR",
    "V1/C3dsR+SA"
]

# ============================================================
# Helper functions
# ============================================================

def find_file(filename, base_dir=Path(".")):
    """
    Find file with or without .csv extension.
    """
    p = base_dir / filename
    if p.exists():
        return p

    p_csv = base_dir / f"{filename}.csv"
    if p_csv.exists():
        return p_csv

    raise FileNotFoundError(
        f"Cannot find {filename} or {filename}.csv in {base_dir.resolve()}"
    )


def read_deg_table(file_path):
    """
    Read DESeq2 annotated table safely.
    Your files contain both 'gene' and 'GeneID', so GeneID is used for overlap.
    """
    df = pd.read_csv(file_path)

    required_cols = {"log2FoldChange", "padj"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{file_path} is missing columns: {missing}")

    # Use cleaned GeneID if available
    if "GeneID" in df.columns:
        df["gene_for_overlap"] = df["GeneID"].astype(str)
    elif "gene" in df.columns:
        df["gene_for_overlap"] = df["gene"].astype(str)
    else:
        df["gene_for_overlap"] = df.iloc[:, 0].astype(str)

    return df


def get_deg_sets(direction="up"):
    """
    Extract upregulated or downregulated DEG sets.
    """
    gene_sets = {}

    for label, fname in deg_files.items():
        file_path = find_file(fname, base_dir)
        df = read_deg_table(file_path)

        df = df.dropna(subset=["padj", "log2FoldChange"]).copy()

        if direction == "up":
            selected = df[
                (df["padj"] < padj_cut) &
                (df["log2FoldChange"] > lfc_cut)
            ].copy()

        elif direction == "down":
            selected = df[
                (df["padj"] < padj_cut) &
                (df["log2FoldChange"] < -lfc_cut)
            ].copy()

        else:
            raise ValueError("direction must be 'up' or 'down'")

        gene_sets[label] = set(selected["gene_for_overlap"].astype(str).tolist())

    gene_sets = {k: gene_sets[k] for k in set_order}

    return gene_sets


def make_membership_table(gene_sets):
    """
    Make gene-level membership table.
    """
    all_genes = sorted(set().union(*gene_sets.values()))

    table = pd.DataFrame({"gene": all_genes})

    for label, genes in gene_sets.items():
        table[label] = table["gene"].isin(genes)

    table["overlap_n"] = table[list(gene_sets.keys())].sum(axis=1)

    def pattern(row):
        return " | ".join([label for label in gene_sets if row[label]])

    table["overlap_pattern"] = table.apply(pattern, axis=1)

    cols = ["gene", "overlap_n", "overlap_pattern"] + list(gene_sets.keys())
    table = table[cols]

    return table


def make_intersection_table(gene_sets):
    """
    Make exclusive UpSet intersection table.
    Each intersection means genes present in exactly those treatments
    and absent from the other treatments shown.
    """
    membership = make_membership_table(gene_sets)
    labels = list(gene_sets.keys())

    rows = []

    for bits in product([False, True], repeat=len(labels)):
        if not any(bits):
            continue

        mask = np.ones(len(membership), dtype=bool)

        for label, bit in zip(labels, bits):
            mask &= membership[label].values == bit

        genes = membership.loc[mask, "gene"].tolist()

        if len(genes) > 0:
            row = {label: bit for label, bit in zip(labels, bits)}
            row["count"] = len(genes)
            row["genes"] = ";".join(genes)
            row["pattern"] = " | ".join(
                [label for label, bit in zip(labels, bits) if bit]
            )
            rows.append(row)

    inter_df = pd.DataFrame(rows)

    if len(inter_df) > 0:
        inter_df = inter_df.sort_values("count", ascending=False).reset_index(drop=True)

    return inter_df


def plot_one_upset(fig, outer_spec, gene_sets, title, panel_label, max_intersections=25):
    """
    Draw one custom UpSet-style plot inside a matplotlib GridSpec slot.
    """
    labels = list(gene_sets.keys())
    set_sizes = pd.Series({label: len(gene_sets[label]) for label in labels})

    inter_df = make_intersection_table(gene_sets)

    if len(inter_df) == 0:
        ax = fig.add_subplot(outer_spec)
        ax.text(0.5, 0.5, "No intersections found", ha="center", va="center")
        ax.set_axis_off()
        return

    inter_df = inter_df.head(max_intersections).copy()

    gs = outer_spec.subgridspec(
        2, 2,
        height_ratios=[1.25, 2.2],
        width_ratios=[1.45, 4.2],
        hspace=0.05,
        wspace=0.04
    )

    ax_blank = fig.add_subplot(gs[0, 0])
    ax_bar = fig.add_subplot(gs[0, 1])
    ax_set = fig.add_subplot(gs[1, 0])
    ax_matrix = fig.add_subplot(gs[1, 1])

    ax_blank.axis("off")

    # --------------------------
    # Top intersection bars
    # --------------------------
    x = np.arange(len(inter_df))
    counts = inter_df["count"].values

    ax_bar.bar(x, counts, color="black", width=0.75)
    ax_bar.set_ylabel("Overlap", fontsize=9)
    ax_bar.set_title(title, fontsize=11, fontweight="bold", pad=6)

    ymax = max(counts)
    for i, c in enumerate(counts):
        ax_bar.text(
            i,
            c + ymax * 0.03,
            str(c),
            ha="center",
            va="bottom",
            fontsize=7
        )

    ax_bar.set_xticks([])
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
    ax_bar.spines["bottom"].set_visible(False)

    # --------------------------
    # Matrix dots
    # --------------------------
    n_sets = len(labels)
    y_positions = np.arange(n_sets)[::-1]

    for idx, y in enumerate(y_positions):
        if idx % 2 == 0:
            ax_matrix.axhspan(y - 0.45, y + 0.45, color="#e8e8ff", zorder=0)

    # pale dots everywhere
    for xi in x:
        ax_matrix.scatter(
            [xi] * n_sets,
            y_positions,
            s=18,
            color="lightgrey",
            zorder=2
        )

    # active dots and connecting lines
    for xi, (_, row) in enumerate(inter_df.iterrows()):
        active_y = []

        for label, y in zip(labels, y_positions):
            if row[label]:
                active_y.append(y)

        if len(active_y) > 1:
            ax_matrix.plot(
                [xi, xi],
                [min(active_y), max(active_y)],
                color="black",
                linewidth=1.1,
                zorder=3
            )

        ax_matrix.scatter(
            [xi] * len(active_y),
            active_y,
            s=24,
            color="black",
            zorder=4
        )

    ax_matrix.set_xlim(-0.7, len(inter_df) - 0.3)
    ax_matrix.set_ylim(-0.6, n_sets - 0.4)
    ax_matrix.set_xticks([])
    ax_matrix.set_yticks(y_positions)
    ax_matrix.set_yticklabels([])
    ax_matrix.tick_params(axis="y", length=0)

    for spine in ["top", "right", "bottom"]:
        ax_matrix.spines[spine].set_visible(False)

    # --------------------------
    # Left set-size bars
    # --------------------------
    bar_values = set_sizes.loc[labels].values
    max_bar = max(bar_values) if max(bar_values) > 0 else 1

    ax_set.barh(
        y_positions,
        bar_values,
        height=0.45,
        color="#d98c80",
        edgecolor="none"
    )

    ax_set.set_yticks(y_positions)
    ax_set.set_yticklabels(labels, fontsize=8)
    ax_set.invert_xaxis()
    ax_set.set_xlabel("DEGs per\ntreatment", fontsize=8)
    ax_set.tick_params(axis="y", length=0)
    ax_set.tick_params(axis="x", labelsize=7)

    ax_set.spines["top"].set_visible(False)
    ax_set.spines["right"].set_visible(False)
    ax_set.spines["left"].set_visible(False)

    for y, v in zip(y_positions, bar_values):
        ax_set.text(
            v + max_bar * 0.03,
            y,
            str(v),
            va="center",
            ha="left",
            fontsize=7
        )

    # Panel label
    ax_bar.text(
        -0.16,
        1.16,
        panel_label,
        transform=ax_bar.transAxes,
        fontsize=14,
        fontweight="bold",
        va="top",
        ha="left"
    )


# ============================================================
# Generate DEG sets
# ============================================================

up_sets = get_deg_sets(direction="up")
down_sets = get_deg_sets(direction="down")

print("Upregulated DEG numbers")
print(pd.Series({k: len(v) for k, v in up_sets.items()}))

print("\nDownregulated DEG numbers")
print(pd.Series({k: len(v) for k, v in down_sets.items()}))

# ============================================================
# Save membership and intersection tables
# ============================================================

up_membership = make_membership_table(up_sets)
down_membership = make_membership_table(down_sets)

up_intersections = make_intersection_table(up_sets)
down_intersections = make_intersection_table(down_sets)

up_membership.to_csv("Upregulated_DEG_overlap_membership.csv", index=False)
down_membership.to_csv("Downregulated_DEG_overlap_membership.csv", index=False)

up_intersections.to_csv("Upregulated_DEG_exclusive_intersections.csv", index=False)
down_intersections.to_csv("Downregulated_DEG_exclusive_intersections.csv", index=False)

# ============================================================
# Plot two-panel figure
# ============================================================

fig = plt.figure(figsize=(14, 7.2))

outer = fig.add_gridspec(
    1, 2,
    width_ratios=[1, 1],
    wspace=0.18
)

plot_one_upset(
    fig,
    outer[0],
    up_sets,
    title=f"Upregulated DEGs\npadj < {padj_cut}, log2FC > {lfc_cut}",
    panel_label=" ",
    max_intersections=25
)

plot_one_upset(
    fig,
    outer[1],
    down_sets,
    title=f"Downregulated DEGs\npadj < {padj_cut}, log2FC < -{lfc_cut}",
    panel_label=" ",
    max_intersections=25
)

plt.savefig("DEG_UpSet_up_down_two_panel.png", dpi=600, bbox_inches="tight")
plt.savefig("DEG_UpSet_up_down_two_panel.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# Settings
# ============================================================

base_dir = Path(".")   # change to Path("counts") if files are in counts/

padj_cut = 0.05
lfc_cut = 1

deg_files = {
    "NSC_vs_Mock": "NSC_vs_Mock_DE_annotated",
    "NSCSA_vs_Mock": "NSCSA_vs_Mock_DE_annotated",
    "SA_vs_Mock": "SA_vs_Mock_DE_annotated",
    "Spec_vs_Mock": "Spec_vs_Mock_DE_annotated",
    "SpecSA_vs_Mock": "SpecSA_vs_Mock_DE_annotated"
}

# Output folder
out_dir = Path("DEG_filtered_padj0.05_log2FC1")
out_dir.mkdir(exist_ok=True)

# ============================================================
# Helper function
# ============================================================

def find_file(filename, base_dir=Path(".")):
    """
    Find file with or without .csv extension.
    """
    p = base_dir / filename
    if p.exists():
        return p

    p_csv = base_dir / f"{filename}.csv"
    if p_csv.exists():
        return p_csv

    raise FileNotFoundError(
        f"Cannot find {filename} or {filename}.csv in {base_dir.resolve()}"
    )

# ============================================================
# Filter and save up/down DEGs
# ============================================================

summary_rows = []

for contrast_name, filename in deg_files.items():

    file_path = find_file(filename, base_dir)
    df = pd.read_csv(file_path)

    # Check required columns
    required_cols = {"log2FoldChange", "padj"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"{file_path} is missing columns: {missing}")

    # Remove rows with NA padj or log2FoldChange
    df_clean = df.dropna(subset=["padj", "log2FoldChange"]).copy()

    # Upregulated genes
    up_df = df_clean[
        (df_clean["padj"] < padj_cut) &
        (df_clean["log2FoldChange"] > lfc_cut)
    ].copy()

    # Downregulated genes
    down_df = df_clean[
        (df_clean["padj"] < padj_cut) &
        (df_clean["log2FoldChange"] < -lfc_cut)
    ].copy()

    # Sort by padj, then absolute log2FC
    up_df = up_df.sort_values(
        by=["padj", "log2FoldChange"],
        ascending=[True, False]
    )

    down_df = down_df.sort_values(
        by=["padj", "log2FoldChange"],
        ascending=[True, True]
    )

    # Save files
    up_out = out_dir / f"{contrast_name}_upregulated_padj0.05_log2FCgt1.csv"
    down_out = out_dir / f"{contrast_name}_downregulated_padj0.05_log2FCltminus1.csv"

    up_df.to_csv(up_out, index=False)
    down_df.to_csv(down_out, index=False)

    # Summary
    summary_rows.append({
        "contrast": contrast_name,
        "total_tested_after_NA_removed": len(df_clean),
        "upregulated": len(up_df),
        "downregulated": len(down_df),
        "total_DEGs": len(up_df) + len(down_df)
    })

    print(f"{contrast_name}")
    print(f"  Upregulated:   {len(up_df)} -> {up_out}")
    print(f"  Downregulated: {len(down_df)} -> {down_out}")

# Save summary table
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(out_dir / "DEG_up_down_summary_padj0.05_log2FC1.csv", index=False)

summary_df